# MLP Leistung ohne optuna

In [2]:
import kagglehub
import pandas
import torch
import matplotlib.pyplot as plt
import numpy

from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PowerTransformer
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Verwendetes Gerät: {device}")

# 1. Datensatz runterladen und mit Pandas einlesen
path = kagglehub.dataset_download("mchilamwar/predict-concrete-strength")
df = pandas.read_csv(f"{path}/ConcreteStrengthData.csv")
df = df.rename(columns={"CementComponent ": "CementComponent"})

# 2. Definiere, was Input x und Output y für das Modell sein soll
zielspalte = "Strength"
x = df.drop( zielspalte, axis="columns")
y = df[ [zielspalte] ]

# 3. Train-Test-Split
x_train, x_test, y_train, y_test = train_test_split(x,y,
                                                    test_size=0.2,
                                                    random_state=123
                                                   )
# 4. Skalierung der Daten
x_scaler = StandardScaler()
y_scaler = StandardScaler()
x_train_scaled = x_scaler.fit_transform(x_train)
y_train_scaled = y_scaler.fit_transform(y_train)

# 5. Erzeuge jetzt aus den Daten Tensoren
x_train_scaled_tensor = torch.tensor(x_train_scaled, dtype=torch.float32)
y_train_scaled_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

# Wohin die Daten schieben? Auf CPU oder auf GPU?
x_train_scaled_tensor = x_train_scaled_tensor.to(device)
y_train_scaled_tensor = y_train_scaled_tensor.to(device)

nr_inputs = x_train.shape[-1]
nr_outputs = y_train.shape[-1]

MAPEs = []
for run in range(5):

    print(run, end=" ")

    # 6. Modell bauen (8-100-50-1 MLP)
    N1 = 100
    N2 = 50
    dropout_rate = 0.1
    model = nn.Sequential(
        nn.Linear(nr_inputs,N1),
        nn.ReLU(),
        nn.Dropout(p=dropout_rate),
        
        nn.Linear(N1,N2),
        nn.ReLU(),
        
        nn.Linear(N2,nr_outputs),
    ).to(device)
        
    # 7. Modell trainieren
    loss_function = nn.MSELoss() # Mittler Quadratischer Fehler = Mean Squared Error
    #loss_function = nn.L1Loss() # MAE
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    model.train()
    losses = []
    for epoch in range(1000):        
        predictions = model( x_train_scaled_tensor )
        loss = loss_function(predictions, y_train_scaled_tensor)
        losses.append(loss.item())
            
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # 8. Das fertige MLP testen
    x_test_scaled = x_scaler.transform(x_test)
    x_test_scaled_tensor = torch.tensor(x_test_scaled, dtype=torch.float32)
    x_test_scaled_tensor = x_test_scaled_tensor.to(device)
    model.eval()
    predictions_test_scaled = model(x_test_scaled_tensor)
    predictions_test = y_scaler.inverse_transform(
        predictions_test_scaled.detach().cpu().numpy())
    y_pred = predictions_test.flatten()
    y_true = y_test["Strength"].values    
    MAPE = mean_absolute_percentage_error(y_true, y_pred)*100
    MAPEs.append( MAPE )

print()
print(MAPEs)
print(f"Average MAPE : {numpy.mean(MAPEs):.1f}%")

# Abschließend ein Vergleich mit Standard-Baseline: MVLR

# 1. Trainiere die Lineare Regression
base_model = LinearRegression().fit(x_train_scaled, y_train_scaled)

# 2. Teste das Modell mit den Testdaten
y_pred_scaled = base_model.predict( x_test_scaled )
y_pred = y_scaler.inverse_transform( y_pred_scaled )

# 3. Berechne den MAPE der LR auf den Testdaten
y_true = y_test["Strength"].values
MAPE = mean_absolute_percentage_error(y_true, y_pred.flatten())*100
print(f"MVLR MAPE: {MAPE:.1f}%")

# Modell und Skalierer abspeichern
import pickle
datei = open("mlp_betonstaerken_regressor.pkl", "wb")
pickle.dump(x_scaler, datei)
pickle.dump(model, datei)
pickle.dump(y_scaler, datei)
datei.close()

Verwendetes Gerät: cuda
0 1 2 3 4 
[10.171435119329738, 9.588015780220822, 9.951585664158108, 10.398915346185454, 9.78480198221821]
Average MAPE : 10.0%
MVLR MAPE: 27.9%


# Systematisches Experimentieren

"Brute Force"

In [ ]:
for N1 in range(10,101,10):
    for N2 in range(10,51,10):
        for lr in range(1,11):
            print(f"{N1=}, {N2=}, lr={lr/100:.2f}")

In [1]:
from itertools import product

N1 = range(10,101,10)
N2 = range(10,51,10)
lr = [0.001, 0.01, 0.1]

i = 0
for x in product(N1, N2, lr):
    print(x)
    i = i + 1
    if i == 20:
        break

(10, 10, 0.001)
(10, 10, 0.01)
(10, 10, 0.1)
(10, 20, 0.001)
(10, 20, 0.01)
(10, 20, 0.1)
(10, 30, 0.001)
(10, 30, 0.01)
(10, 30, 0.1)
(10, 40, 0.001)
(10, 40, 0.01)
(10, 40, 0.1)
(10, 50, 0.001)
(10, 50, 0.01)
(10, 50, 0.1)
(20, 10, 0.001)
(20, 10, 0.01)
(20, 10, 0.1)
(20, 20, 0.001)
(20, 20, 0.01)


# Optimierung mit optuna

In [5]:
import kagglehub
import pandas
import torch
import matplotlib.pyplot as plt
import numpy

from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PowerTransformer
from sklearn.metrics import mean_absolute_percentage_error

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Verwendetes Gerät: {device}")

# 1. Datensatz runterladen und mit Pandas einlesen
path = kagglehub.dataset_download("mchilamwar/predict-concrete-strength")
df = pandas.read_csv(f"{path}/ConcreteStrengthData.csv")
df = df.rename(columns={"CementComponent ": "CementComponent"})

# 2. Definiere, was Input x und Output y für das Modell sein soll
zielspalte = "Strength"
x = df.drop( zielspalte, axis="columns")
y = df[ [zielspalte] ]

# 3. Train-Test-Split
x_train, x_test, y_train, y_test = train_test_split(x,y,
                                                    test_size=0.2,
                                                    random_state=123
                                                   )
# 4. Skalierung der Daten
x_scaler = StandardScaler()
y_scaler = StandardScaler()
x_train_scaled = x_scaler.fit_transform(x_train)
y_train_scaled = y_scaler.fit_transform(y_train)

# 5. Erzeuge jetzt aus den Daten Tensoren
x_train_scaled_tensor = torch.tensor(x_train_scaled, dtype=torch.float32)
y_train_scaled_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

# Wohin die Daten schieben? Auf CPU oder auf GPU?
x_train_scaled_tensor = x_train_scaled_tensor.to(device)
y_train_scaled_tensor = y_train_scaled_tensor.to(device)

nr_inputs = x_train.shape[-1]
nr_outputs = y_train.shape[-1]


# 6. Modell bauen (8-100-50-1 MLP)
N1 = 100
N2 = 50
dropout_rate = 0.1
model = nn.Sequential(
    nn.Linear(nr_inputs,N1),
    nn.ReLU(),
    nn.Dropout(p=dropout_rate),
    
    nn.Linear(N1,N2),
    nn.ReLU(),
    
    nn.Linear(N2,nr_outputs),
).to(device)
    
# 7. Modell trainieren
loss_function = nn.MSELoss() # Mittler Quadratischer Fehler = Mean Squared Error
#loss_function = nn.L1Loss() # MAE
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
model.train()
losses = []
for epoch in range(1000):        
    predictions = model( x_train_scaled_tensor )
    loss = loss_function(predictions, y_train_scaled_tensor)
    losses.append(loss.item())
        
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# 8. Das fertige MLP testen
x_test_scaled = x_scaler.transform(x_test)
x_test_scaled_tensor = torch.tensor(x_test_scaled, dtype=torch.float32)
x_test_scaled_tensor = x_test_scaled_tensor.to(device)
model.eval()
predictions_test_scaled = model(x_test_scaled_tensor)
predictions_test = y_scaler.inverse_transform(
    predictions_test_scaled.detach().cpu().numpy())
y_pred = predictions_test.flatten()
y_true = y_test["Strength"].values    
MAPE = mean_absolute_percentage_error(y_true, y_pred)*100
print(f"MAPE={MAPE:.2f}%")

Verwendetes Gerät: cuda
MAPE=9.67%


In [6]:
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]2m2/3 [optuna]
Note: you may need to restart the kernel to use updated packages.


In [8]:
# Falls Optuna noch fehlt: einmal %pip install optuna im Notebook ausführen.
import optuna
from sklearn.preprocessing import FunctionTransformer

# Voraussetzung: die vorherige Zelle mit dem Train-Test-Split wurde ausgeführt.
# Nur Trainingsdaten für die Suche verwenden; Testdaten bleiben außen vor.
x_fit, x_val, y_fit, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=123
)
SEED = 123
MAX_EPOCHS = 1000


def make_scaler(name):
    return {
        "keine": lambda: FunctionTransformer(validate=True),
        "standard": StandardScaler,
        "minmax": MinMaxScaler,
        "power": lambda: PowerTransformer(method="yeo-johnson"),
    }[name]()


def as_tensor(data):
    return torch.as_tensor(numpy.asarray(data), dtype=torch.float32, device=device)


def make_model(params):
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    activation = {"ReLU": nn.ReLU, "Tanh": nn.Tanh, "ELU": nn.ELU,
                  "LeakyReLU": nn.LeakyReLU}[params["activation"]]
    layers = []
    n_in = x_train.shape[1]
    for i in range(params["n_layers"]):
        n_out = params[f"n_units_{i}"]
        layers.extend([nn.Linear(n_in, n_out), activation()])
        if i == 0:
            layers.append(nn.Dropout(p=0.1))  # wie im bisherigen MLP
        n_in = n_out
    layers.append(nn.Linear(n_in, y_train.shape[1]))
    return nn.Sequential(*layers).to(device)


def make_optimizer(model, params):
    optimizer_class = {"Adam": torch.optim.Adam, "RMSprop": torch.optim.RMSprop,
                       "SGD": torch.optim.SGD}[params["optimizer"]]
    kwargs = {"momentum": 0.9} if params["optimizer"] == "SGD" else {}
    return optimizer_class(model.parameters(), lr=params["lr"], **kwargs)


def objective(trial):
    params = {
        "n_layers": trial.suggest_int("n_layers", 1, 4),
        "activation": trial.suggest_categorical(
            "activation", ["ReLU", "Tanh", "ELU", "LeakyReLU"]),
        "x_scaling": trial.suggest_categorical(
            "x_scaling", ["keine", "standard", "minmax", "power"]),
        # Affine Zielskalierung: Rücktransformation ist für alle Vorhersagen definiert.
        "y_scaling": trial.suggest_categorical(
            "y_scaling", ["keine", "standard", "minmax"]),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-1, log=True),
    }
    for i in range(params["n_layers"]):
        params[f"n_units_{i}"] = trial.suggest_int(f"n_units_{i}", 16, 256, step=16)

    # Skalierer ausschließlich auf dem inneren Trainingssatz fitten (kein Leakage).
    sx, sy = make_scaler(params["x_scaling"]), make_scaler(params["y_scaling"])
    xt, yt = as_tensor(sx.fit_transform(x_fit)), as_tensor(sy.fit_transform(y_fit))
    xv = as_tensor(sx.transform(x_val))
    net = make_model(params)
    optimizer = make_optimizer(net, params)
    loss_function = nn.MSELoss()
    best_mape = float("inf")

    for epoch in range(1, MAX_EPOCHS + 1):
        net.train()
        optimizer.zero_grad()
        loss = loss_function(net(xt), yt)
        if not torch.isfinite(loss).item():
            raise optuna.TrialPruned("Training numerisch instabil")
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            net.eval()
            with torch.no_grad():
                pred = sy.inverse_transform(net(xv).cpu().numpy())
            if not numpy.isfinite(pred).all():
                raise optuna.TrialPruned("Vorhersagen numerisch instabil")
            mape = mean_absolute_percentage_error(y_val, pred) * 100
            if mape < best_mape:
                best_mape = float(mape)
                trial.set_user_attr("best_epoch", epoch)
            trial.report(best_mape, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
    return best_mape


# 200 Versuche insgesamt, einschließlich frühzeitig abgebrochener Trials.
# API: https://optuna.readthedocs.io/en/stable/reference/generated/optuna.study.Study.html
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=100),
)
study.optimize(objective, n_trials=1000, n_jobs=1, show_progress_bar=True)

print(f"Bester Validierungs-MAPE: {study.best_value:.2f}%")
print("Beste Hyperparameter:", study.best_params)

# Gewinner auf allen Trainingsdaten neu trainieren. Die Epochenzahl stammt
# ebenfalls aus der Validierung; die Testdaten bestimmen keine Einstellungen.
best_params = study.best_params
best_epochs = study.best_trial.user_attrs["best_epoch"]
best_x_scaler = make_scaler(best_params["x_scaling"])
best_y_scaler = make_scaler(best_params["y_scaling"])
xt = as_tensor(best_x_scaler.fit_transform(x_train))
yt = as_tensor(best_y_scaler.fit_transform(y_train))
best_model = make_model(best_params)
best_optimizer = make_optimizer(best_model, best_params)
loss_function = nn.MSELoss()
best_model.train()
for epoch in range(best_epochs):
    best_optimizer.zero_grad()
    loss = loss_function(best_model(xt), yt)
    if not torch.isfinite(loss).item():
        raise RuntimeError("Finales Training instabil; Testbewertung wird abgebrochen.")
    loss.backward()
    best_optimizer.step()

best_model.eval()
with torch.no_grad():
    predictions_optuna = best_y_scaler.inverse_transform(
        best_model(as_tensor(best_x_scaler.transform(x_test))).cpu().numpy()
    )
if not numpy.isfinite(predictions_optuna).all():
    raise RuntimeError("Finale Vorhersagen enthalten nicht-endliche Werte.")
test_mape_optuna = mean_absolute_percentage_error(y_test, predictions_optuna) * 100
print(f"Finales Training: {best_epochs} Epochen")
print(f"Test-MAPE des Optuna-Modells: {test_mape_optuna:.2f}%")
# Für spätere Vorhersagen: best_x_scaler -> best_model -> best_y_scaler.inverse_transform


[I 2026-10-23 07:43:38,070] A new study created in memory with name: no-name-ce908662-6382-4358-a248-ab3a107f3478


  0%|          | 0/1000 [00:00<?, ?it/s]

[I 2026-10-23 07:43:39,068] Trial 0 finished with value: 9.38643217086792 and parameters: {'n_layers': 3, 'activation': 'LeakyReLU', 'x_scaling': 'standard', 'y_scaling': 'minmax', 'optimizer': 'Adam', 'lr': 0.008953268791955234, 'n_units_0': 48, 'n_units_1': 48, 'n_units_2': 144}. Best is trial 0 with value: 9.38643217086792.
[I 2026-10-23 07:43:39,893] Trial 1 finished with value: 27.046582102775574 and parameters: {'n_layers': 3, 'activation': 'Tanh', 'x_scaling': 'keine', 'y_scaling': 'standard', 'optimizer': 'SGD', 'lr': 0.0005050346330980692, 'n_units_0': 80, 'n_units_1': 112, 'n_units_2': 240}. Best is trial 0 with value: 9.38643217086792.
[I 2026-10-23 07:43:40,906] Trial 2 finished with value: 12.317173928022385 and parameters: {'n_layers': 4, 'activation': 'Tanh', 'x_scaling': 'standard', 'y_scaling': 'keine', 'optimizer': 'RMSprop', 'lr': 0.0015145095837218216, 'n_units_0': 96, 'n_units_1': 80, 'n_units_2': 112, 'n_units_3': 176}. Best is trial 0 with value: 9.38643217086792